# 03 — Incremental Load Validation

This notebook validates the incremental extraction flow from FIPEX GitHub Releases.

## Current scope

Implemented and tested so far:

- Resolve FIPEX monthly release
- Select the original unmerged Parquet asset
- Download the full release snapshot temporarily
- Filter the requested reference month
- Persist one monthly Bronze Parquet
- Validate period integrity
- Validate idempotency
- Inspect local Bronze coverage
- List remote releases
- Prepare missing-period / catch-up logic

This notebook intentionally stops at the current project stage.


In [1]:
from pathlib import Path
import pandas as pd

from fipe_pipeline.extract import (
    extract_month,
    inspect_local_bronze,
    list_available_releases,
)

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT


WindowsPath('E:/VSCODE Files/Projects/02_fipe_data_pipeline')

## Validate one known published month


In [2]:
result = extract_month(2026, 9)
result


ExtractionResult(year=2026, month=9, release_tag='v2026.09.0', release_url='https://github.com/fipex-labs/dataset/releases/tag/v2026.09.0', asset_name='existing_local_file', asset_download_url='', destination=WindowsPath('E:/VSCODE Files/Projects/02_fipe_data_pipeline/data/bronze/monthly/fipe_2026_09.parquet'), status='already_exists', rows=51012, size_bytes=1131892)

Expected behavior after the first successful download:

```text
status='already_exists'
rows=51012
destination=.../data/bronze/monthly/fipe_2026_09.parquet
```

The first execution would have returned `status='downloaded'`.


In [3]:
monthly_path = PROJECT_ROOT / "data" / "bronze" / "monthly" / "fipe_2026_09.parquet"
monthly_path.exists(), monthly_path


(True,
 WindowsPath('E:/VSCODE Files/Projects/02_fipe_data_pipeline/data/bronze/monthly/fipe_2026_09.parquet'))

In [4]:
df_2026_09 = pd.read_parquet(monthly_path)
df_2026_09.shape


(51012, 12)

In [5]:
df_2026_09[["ano_referencia", "mes_referencia"]].drop_duplicates()


,ano_referencia,mes_referencia
0,2026,9


## Inspect local Bronze coverage


In [6]:
inventory = inspect_local_bronze()
inventory


LocalInventory(historical_watermark=Period(year=2026, month=8), monthly_periods=(Period(year=2026, month=9),))

Expected current local state:

```text
historical watermark = 2026-08
monthly periods      = 2026-09
```


## List FIPEX releases


In [7]:
releases = list_available_releases()
[(release.period.label, release.tag) for release in releases[-10:]]


[('2026-07', 'v2026.07.0'),
 ('2026-08', 'v2026.08.0'),
 ('2026-09', 'v2026.09.0')]

## Next implementation checkpoint

The next step is to finish and validate the automatic catch-up flow:

```text
local historical watermark
        +
local monthly inventory
        +
remote FIPEX releases
        ↓
missing periods
        ↓
chronological extraction
```

The pipeline must not silently skip an intermediate month if a newer release exists.


In [8]:
from fipe_pipeline.extract import (
    inspect_local_bronze,
    list_available_releases,
    find_missing_periods,
    validate_no_missing_remote_gap,
    extract_missing_months,
)

In [9]:
inventory = inspect_local_bronze()

inventory

LocalInventory(historical_watermark=Period(year=2026, month=8), monthly_periods=(Period(year=2026, month=9),))

In [10]:
releases = list_available_releases()

[(r.period.label, r.tag) for r in releases[-10:]]

[('2026-07', 'v2026.07.0'),
 ('2026-08', 'v2026.08.0'),
 ('2026-09', 'v2026.09.0')]

In [11]:
missing = find_missing_periods(
    releases,
    inventory,
)

[(r.period.label, r.tag) for r in missing]

[]

In [12]:
validate_no_missing_remote_gap(
    inventory,
    releases,
)

In [13]:
catchup = extract_missing_months()

catchup

CatchUpResult(historical_watermark=Period(year=2026, month=8), local_periods_before=(Period(year=2026, month=9),), available_periods=(Period(year=2026, month=7), Period(year=2026, month=8), Period(year=2026, month=9)), missing_periods=(), extraction_results=())

In [14]:
inventory = inspect_local_bronze()
inventory

LocalInventory(historical_watermark=Period(year=2026, month=8), monthly_periods=(Period(year=2026, month=9),))

In [15]:
missing = find_missing_periods(
    releases,
    inventory,
)

[(r.period.label, r.tag) for r in missing]

[]

In [16]:
catchup = extract_missing_months()

catchup

CatchUpResult(historical_watermark=Period(year=2026, month=8), local_periods_before=(Period(year=2026, month=9),), available_periods=(Period(year=2026, month=7), Period(year=2026, month=8), Period(year=2026, month=9)), missing_periods=(), extraction_results=())

In [17]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MONTHLY_PATH = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "monthly"
    / "fipe_2026_09.parquet"
)

df_2026_09 = pd.read_parquet(MONTHLY_PATH)

df_2026_09.shape

(51012, 12)

In [18]:
from fipe_pipeline.validate import run_validations

validation_report_2026_09 = run_validations(df_2026_09)

validation_report_2026_09

,rule,passed,invalid_rows,severity,action,effective_action,message
0,DQ-SCHEMA-001,True,0,ERROR,FAIL_PIPELINE,NONE,All required columns are present.
1,DQ-NULL-001,True,0,ERROR,QUARANTINE,NONE,No unexpected nulls found.
2,DQ-NULL-002,True,0,ERROR,QUARANTINE,NONE,zero_km and ano_modelo are consistent.
3,DQ-TIME-001,True,0,ERROR,QUARANTINE,NONE,All reference months are valid.
4,DQ-TIME-002,True,0,ERROR,QUARANTINE,NONE,All model years satisfy ano_modelo <= ano_refe...
5,DQ-CODE-001,True,0,ERROR,QUARANTINE,NONE,All codigo_fipe values match the expected format.
6,DQ-PRICE-001,True,0,ERROR,QUARANTINE,NONE,All prices are positive.
7,DQ-PRICE-002,True,0,ERROR,QUARANTINE,NONE,valor_formatado matches valor_centavos.
8,DQ-DUP-001,True,0,WARNING,DEDUPLICATE,NONE,No exact duplicate rows found.
9,DQ-GRAIN-001,True,0,ERROR,QUARANTINE,NONE,No non-exact logical grain collisions found.


In [19]:
from fipe_pipeline.transform import transform_bronze_to_silver

result_2026_09 = transform_bronze_to_silver(df_2026_09)

In [20]:
print("Silver:", result_2026_09.silver.shape)
print("Quarantine:", result_2026_09.quarantine.shape)
print("Duplicates:", result_2026_09.duplicates.shape)

Silver: (51012, 14)
Quarantine: (0, 17)
Duplicates: (0, 17)


In [21]:
result_2026_09.quarantine["dq_reasons"].value_counts()

Series([], Name: count, dtype: int64)

In [22]:
result_2026_09.duplicates["dq_reasons"].value_counts()

Series([], Name: count, dtype: int64)

In [23]:
result_2026_09.silver[
    ["ano_referencia", "mes_referencia"]
].drop_duplicates()

,ano_referencia,mes_referencia
0,2026,9


In [25]:
from pathlib import Path

import pandas as pd

from fipe_pipeline.transform import transform_bronze_to_silver
from fipe_pipeline.load import load_historical_transform_result

In [26]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

HISTORICAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "historical"
    / "fipe_history_2026_08.parquet"
)

df_history = pd.read_parquet(HISTORICAL_PATH)

In [27]:
result_history = transform_bronze_to_silver(
    df_history
)

In [28]:
print("Silver:", result_history.silver.shape)
print("Quarantine:", result_history.quarantine.shape)
print("Duplicates:", result_history.duplicates.shape)

Silver: (9477932, 14)
Quarantine: (190, 17)
Duplicates: (83, 17)


In [30]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "data" / "silver"

silver_files = sorted(
    SILVER_DIR.glob("year=*/month=*/fipe.parquet")
)

len(silver_files)

309

In [31]:
silver_inventory = []

for path in silver_files:
    df = pd.read_parquet(
        path,
        columns=[
            "ano_referencia",
            "mes_referencia",
        ],
    )

    periods = (
        df[
            ["ano_referencia", "mes_referencia"]
        ]
        .drop_duplicates()
    )

    if len(periods) != 1:
        raise ValueError(
            f"Partition with multiple periods: {path}"
        )

    row = periods.iloc[0]

    silver_inventory.append(
        {
            "year": int(row["ano_referencia"]),
            "month": int(row["mes_referencia"]),
            "rows": len(df),
            "path": path,
        }
    )

silver_inventory = pd.DataFrame(
    silver_inventory
).sort_values(
    ["year", "month"]
).reset_index(drop=True)

silver_inventory

,year,month,rows,path
0,2001,1,11844,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
1,2001,2,11660,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
2,2001,3,11107,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
3,2001,4,12171,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
4,2001,5,12566,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
...,...,...,...,...
304,2026,5,50252,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
305,2026,6,50395,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
306,2026,7,50599,E:\VSCODE Files\Projects\02_fipe_data_pipeline...
307,2026,8,50838,E:\VSCODE Files\Projects\02_fipe_data_pipeline...


In [32]:
silver_inventory["data_referencia"] = pd.to_datetime(
    {
        "year": silver_inventory["year"],
        "month": silver_inventory["month"],
        "day": 1,
    }
)

expected_periods = pd.date_range(
    silver_inventory["data_referencia"].min(),
    silver_inventory["data_referencia"].max(),
    freq="MS",
)

missing_periods = expected_periods.difference(
    silver_inventory["data_referencia"]
)

missing_periods

DatetimeIndex([], dtype='datetime64[us]', freq='MS')

In [33]:
silver_inventory[
    ["year", "month"]
].iloc[[0, -1]]

,year,month
0,2001,1
308,2026,9


In [34]:
silver_inventory["rows"].sum()

np.int64(9528944)

In [35]:
silver_inventory.duplicated(
    subset=["year", "month"]
).sum()

np.int64(0)

In [36]:
from fipe_pipeline.gold import build_gold

gold_result = build_gold()

gold_result

GoldBuildResult(source_partitions=309, rows=9528944, first_period=(2001, 1), last_period=(2026, 9), destination=WindowsPath('E:/VSCODE Files/Projects/02_fipe_data_pipeline/data/gold/fipe_prices.parquet'), size_bytes=141000730)

In [37]:
from fipe_pipeline.pipeline import run_pipeline

pipeline_result = run_pipeline()

pipeline_result

PipelineRunResult(extracted_months=(), processed_months=(), gold_result=None)

In [38]:
from fipe_pipeline.logging_config import configure_logging
from fipe_pipeline.pipeline import run_pipeline

configure_logging()

result = run_pipeline()

result

2026-09-17 11:17:58 | INFO | fipe_pipeline.pipeline | Starting incremental FIPE pipeline.
2026-09-17 11:18:00 | INFO | fipe_pipeline.pipeline | Pending Bronze months for Silver processing: []
2026-09-17 11:18:00 | INFO | fipe_pipeline.pipeline | Incremental FIPE pipeline completed.


PipelineRunResult(extracted_months=(), processed_months=(), gold_result=None)

# DuckDB Validation Layer

In [1]:
from fipe_pipeline.duckdb_layer import (
    connect_duckdb,
    register_parquet_views,
    validate_duckdb_layer,
)

In [2]:
con = connect_duckdb()

register_parquet_views(con)

In [3]:
duckdb_validation = validate_duckdb_layer(con)

duckdb_validation

DuckDBValidationResult(silver_rows=9528944, gold_rows=9528944, row_counts_match=True, silver_first_period=(2001, 1), silver_last_period=(2026, 9), gold_first_period=(2001, 1), gold_last_period=(2026, 9), periods_match=True)

In [4]:
con.close()